In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.8 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/abewley/sort.git

Cloning into 'sort'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 208 (delta 45), reused 40 (delta 40), pack-reused 159 (from 1)
Receiving objects: 100% (208/208), 1.20 MiB | 7.96 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [ ]:
import re

# Read the content of sort.py
with open('/content/sort/sort.py', 'r') as f:
    sort_file_content = f.read()

# Replace matplotlib.use('TkAgg') with matplotlib.use('Agg')
modified_sort_file_content = re.sub(r"matplotlib.use\('TkAgg'\)", "matplotlib.use('Agg')", sort_file_content)

# Write the modified content back to sort.py
with open('/content/sort/sort.py', 'w') as f:
    f.write(modified_sort_file_content)

print("Modified sort.py to use 'Agg' matplotlib backend.")

Modified sort.py to use 'Agg' matplotlib backend.


In [ ]:
!pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=6fcc3109fdad87511b842f6b758824909c4fd7b375c407b83fe4dbf824086262
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from sort.sort import Sort  # Corrected import statement
import gradio as gr
import threading
import time
from PIL import Image
import os
import torch

# --- Global variables for the monitoring system ---
model = None
caps = {}
trackers = {}
vehicle_counts = {}
crossed_vehicles = {}
prev_positions = {}
monitoring_active = False
current_frame = None
frame_lock = threading.Lock()
monitor_thread = None

# --- Global variables for Traffic Light Control ---
queue_lengths = {"North": 0, "East": 0, "West": 0, "South": 0}
light_states = {"North": "green", "East": "red", "West": "red", "South": "red"}
green_light_timer = 0
current_green_direction = "North"

# --- NEW: Amber phase variables ---
amber_timer = 0
next_green_direction = None
total_light_cycles = 0
green_duration_history = []
start_time = None

# --- Traffic Light Timing Constants ---
MIN_GREEN_TIME = 150  # 5 seconds
MAX_GREEN_TIME = 300  # 10 seconds
AMBER_DURATION = 60  # 2 seconds

# --- Traffic Light Color Constants ---
LIGHT_COLORS = {
    "red": (0, 0, 255),      # BGR: Red
    "amber": (0, 165, 255),  # BGR: Orange/Amber
    "green": (0, 255, 0),    # BGR: Green
    "dim": (50, 50, 50)      # BGR: Dim gray for inactive lights
}

# --- Vehicle Detection and Tracking Constants ---
vehicle_classes = ["car", "bus", "truck", "motorbike"]
LINE_POSITION_RATIO = 0.50
LINE_THICKNESS = 3
CROSSING_TOLERANCE = 10
DETECTION_BUFFER = 50

def initialize_model():
    """Load the YOLOv8 model and prepare it for the available device (GPU/CPU)."""
    global model
    try:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {device}")

        # Try loading different model sizes, starting with balanced speed/accuracy
        model_options = ["yolov8s.pt", "yolov8m.pt"]
        for model_name in model_options:
            try:
                print(f"Trying to load {model_name}...")
                model = YOLO(model_name)
                print(f"Successfully loaded {model_name}!")
                break
            except Exception as e:
                print(f"Failed to load {model_name}: {e}")
                if model_name == model_options[-1]:
                    raise Exception("Failed to load any YOLO model.")

        model.to(device)
        print("Model initialized successfully!")
        return True
    except Exception as e:
        print(f"Error initializing model: {e}")
        return False

def setup_videos(video_paths_dict):
    """Initialize video captures, trackers, and counters for all video streams."""
    global caps, trackers, vehicle_counts, crossed_vehicles, prev_positions, queue_lengths
    global light_states, green_light_timer, current_green_direction
    global amber_timer, next_green_direction, total_light_cycles, green_duration_history, start_time

    try:
        caps = {name: cv2.VideoCapture(path) for name, path in video_paths_dict.items()}
        for name, cap in caps.items():
            if not cap.isOpened():
                print(f"Error: Could not open video file for {name}")
                return False

        trackers = {name: Sort() for name in video_paths_dict.keys()}
        vehicle_counts = {name: 0 for name in video_paths_dict.keys()}
        crossed_vehicles = {name: set() for name in video_paths_dict.keys()}
        prev_positions = {name: {} for name in video_paths_dict.keys()}

        # Reset traffic light state
        queue_lengths = {name: 0 for name in video_paths_dict.keys()}
        light_states = {"North": "green", "East": "red", "West": "red", "South": "red"}
        green_light_timer = 0
        current_green_direction = "North"

        # Reset amber phase variables
        amber_timer = 0
        next_green_direction = None
        total_light_cycles = 0
        green_duration_history = []
        start_time = time.time()

        print("Video setup completed successfully!")
        return True
    except Exception as e:
        print(f"Error setting up videos: {e}")
        return False

def draw_traffic_light(frame, x, y, state, direction_name, timer):
    """
    Draws a realistic 3-light traffic signal (vertical layout)
    - Red light on top
    - Amber/Yellow light in middle
    - Green light on bottom

    Parameters:
    - frame: The image to draw on
    - x, y: top-left corner position
    - state: "red", "amber", or "green"
    - direction_name: "N", "E", "W", "S"
    - timer: current countdown value in seconds
    """
    # Traffic light housing dimensions
    housing_width = 60
    housing_height = 160
    light_radius = 20
    padding = 20

    # Draw black housing background
    cv2.rectangle(frame, (x, y), (x + housing_width, y + housing_height), (30, 30, 30), -1)
    cv2.rectangle(frame, (x, y), (x + housing_width, y + housing_height), (200, 200, 200), 2)

    # Calculate center positions for each light
    center_x = x + housing_width // 2
    red_y = y + padding + light_radius
    amber_y = y + housing_height // 2
    green_y = y + housing_height - padding - light_radius

    # Determine which lights are lit
    red_color = LIGHT_COLORS["red"] if state == "red" else LIGHT_COLORS["dim"]
    amber_color = LIGHT_COLORS["amber"] if state == "amber" else LIGHT_COLORS["dim"]
    green_color = LIGHT_COLORS["green"] if state == "green" else LIGHT_COLORS["dim"]

    # Draw the three lights
    cv2.circle(frame, (center_x, red_y), light_radius, red_color, -1)
    cv2.circle(frame, (center_x, red_y), light_radius, (100, 100, 100), 2)

    cv2.circle(frame, (center_x, amber_y), light_radius, amber_color, -1)
    cv2.circle(frame, (center_x, amber_y), light_radius, (100, 100, 100), 2)

    cv2.circle(frame, (center_x, green_y), light_radius, green_color, -1)
    cv2.circle(frame, (center_x, green_y), light_radius, (100, 100, 100), 2)

    # Draw direction label
    label_y = y + housing_height + 25
    cv2.putText(frame, direction_name, (center_x - 10, label_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Draw countdown timer
    if state == "green" or state == "amber":
        timer_text = f"{int(timer)}s"
        timer_color = (0, 255, 255)  # Cyan
    else:
        timer_text = "WAIT"
        timer_color = (100, 100, 100)  # Gray

    timer_y = label_y + 25
    cv2.putText(frame, timer_text, (center_x - 20, timer_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, timer_color, 2)

def update_traffic_lights():
    """
    Updates the traffic light states with Round-Robin + Queue Priority.
    Ensures fairness while remaining adaptive to high traffic.
    """
    global green_light_timer, current_green_direction, light_states
    global amber_timer, next_green_direction, total_light_cycles, green_duration_history

    # If we're in amber phase, just count down
    if amber_timer > 0:
        amber_timer += 1
        if amber_timer >= AMBER_DURATION:
            # Amber phase complete - switch to red and activate next green
            light_states[current_green_direction] = "red"
            light_states[next_green_direction] = "green"

            # Record statistics
            total_light_cycles += 1
            green_duration_history.append(green_light_timer / 30.0)  # Convert frames to seconds

            # Update current direction and reset timers
            current_green_direction = next_green_direction
            next_green_direction = None
            amber_timer = 0
            green_light_timer = 0
        return

    # Normal green phase - increment timer
    green_light_timer += 1
    time_to_change = False

    current_queue = queue_lengths[current_green_direction]
    max_other_queue = max([q for d, q in queue_lengths.items() if d != current_green_direction], default=0)

    # Condition 1: Max green time has been exceeded
    if green_light_timer > MAX_GREEN_TIME:
        time_to_change = True
    # Condition 2: Current green lane is empty while others are waiting (after minimum time)
    elif green_light_timer > MIN_GREEN_TIME:
        if current_queue == 0 and sum(queue_lengths.values()) > 0:
            time_to_change = True
    # Condition 3: EMERGENCY OVERRIDE - Another direction has EXTREME congestion
    elif green_light_timer > 60:  # After just 2 seconds (60 frames)
        queue_difference = max_other_queue - current_queue
        # Only switch early if difference is extreme (≥10 vehicles)
        if queue_difference >= 10:
            time_to_change = True

    if time_to_change:
        # Start amber phase
        light_states[current_green_direction] = "amber"
        amber_timer = 1  # Start counting amber phase

        # NEW: Round-Robin with Queue Priority Algorithm
        directions = ["North", "East", "South", "West"]
        current_idx = directions.index(current_green_direction)

        # Default: Next direction in round-robin sequence
        next_in_sequence = directions[(current_idx + 1) % 4]

        # Check if emergency override is needed
        if sum(queue_lengths.values()) == 0:
            # No vehicles anywhere, just follow sequence
            next_green_direction = next_in_sequence
        else:
            # Check for EXTREME congestion (≥10 vehicles difference)
            max_queue = max(queue_lengths.values())
            queue_difference = max_queue - queue_lengths[next_in_sequence]

            if queue_difference >= 10:
                # Emergency: Skip to most congested direction
                next_green_direction = max(queue_lengths, key=queue_lengths.get)
            else:
                # Normal: Follow round-robin sequence (ensures fairness)
                next_green_direction = next_in_sequence

def process_frame():
    """Process a single frame from all video sources, run detection/tracking, and update visualization."""
    global current_frame, queue_lengths

    frames = []
    all_ended = True

    for name, cap in caps.items():
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((360, 640, 3), dtype=np.uint8)
            cv2.putText(frame, f"{name} - Ended", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        else:
            all_ended = False

            # Resize frame BEFORE processing to improve performance
            original_height, original_width = frame.shape[:2]
            if original_width > 1280:
                # Resize to 720p for processing
                frame = cv2.resize(frame, (1280, 720))

            frame_height, frame_width = frame.shape[:2]
            counting_line_y = int(frame_height * LINE_POSITION_RATIO)

            # Create ROI for detection
            roi_frame = frame[0:counting_line_y + DETECTION_BUFFER, :]

            # Run YOLO detection with balanced confidence threshold
            results = model.predict(roi_frame, conf=0.55, verbose=False, device='cuda' if torch.cuda.is_available() else 'cpu', half=torch.cuda.is_available())

            detections = []
            if results and results[0].boxes is not None:
                for box in results[0].boxes:
                    cls_name = model.names[int(box.cls[0])]
                    if cls_name in vehicle_classes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        detections.append([x1, y1, x2, y2, float(box.conf[0])])

            detections_np = np.array(detections) if detections else np.empty((0, 5))

            # Update SORT tracker
            tracks = trackers[name].update(detections_np)
            queue_lengths[name] = len(tracks) # Update queue length

            current_positions = {}
            for tr in tracks:
                x1, y1, x2, y2, track_id = map(int, tr)
                bottom_y = y2
                current_positions[track_id] = bottom_y

                if track_id not in prev_positions[name]:
                    prev_positions[name][track_id] = bottom_y

                prev_y = prev_positions[name][track_id]

                # Line crossing detection logic
                if (prev_y < counting_line_y <= bottom_y) and (track_id not in crossed_vehicles[name]):
                    vehicle_counts[name] += 1
                    crossed_vehicles[name].add(track_id)

                # Visualization
                box_color = (0, 0, 255) if track_id in crossed_vehicles[name] else (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(frame, f"ID {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)

            prev_positions[name].update(current_positions)

            # Draw counting line
            cv2.line(frame, (0, counting_line_y), (frame_width, counting_line_y), (0, 0, 255), LINE_THICKNESS)

        # Resize frame to standard size BEFORE drawing text and lights
        frame = cv2.resize(frame, (640, 360))

        # Now draw text and lights on the resized frame for consistency
        if ret:  # Only draw for active videos
            # Add text background rectangle for better readability
            text = f"{name} - Count: {vehicle_counts[name]} | Queue: {queue_lengths[name]}"
            (text_width, text_height), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
            cv2.rectangle(frame, (5, 5), (15 + text_width, 40), (0, 0, 0), -1)
            cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            # Draw traffic light in bottom-right corner (on resized 640x360 frame)
            light_padding = 30
            light_x = 640 - 90  # Fixed position on 640px width
            light_y = 360 - 220  # Fixed position on 360px height

            # Calculate countdown timer for this direction
            if light_states[name] == "green":
                remaining_frames = MAX_GREEN_TIME - green_light_timer
                timer_value = max(0, remaining_frames / 30.0)
            elif light_states[name] == "amber":
                remaining_frames = AMBER_DURATION - amber_timer
                timer_value = max(0, remaining_frames / 30.0)
            else:
                timer_value = 0

            # Draw the traffic light
            draw_traffic_light(frame, light_x, light_y, light_states[name], name[0], timer_value)

        frames.append(frame)

    # Run the traffic light logic after processing all frames
    update_traffic_lights()

    # Combine frames into a 2x2 grid
    if len(frames) >= 4:
        top = np.hstack((frames[0], frames[1]))    # North, East
        bottom = np.hstack((frames[2], frames[3])) # West, South
        combined = np.vstack((top, bottom))

        # Add total summary at bottom
        total_counted = sum(vehicle_counts.values())

        # Status text with background for readability
        if amber_timer > 0:
            status = f"AMBER PHASE: {current_green_direction} -> {next_green_direction} | TOTAL: {total_counted}"
            status_color = (0, 165, 255)
        else:
            status = f"GREEN: {current_green_direction} | Time: {green_light_timer/30:.1f}s | TOTAL: {total_counted}"
            status_color = (0, 255, 0)

        # Draw status background
        (status_width, status_height), baseline = cv2.getTextSize(status, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(combined, (5, combined.shape[0] - 45), (15 + status_width, combined.shape[0] - 5), (0, 0, 0), -1)
        cv2.putText(combined, status, (10, combined.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        with frame_lock:
            current_frame = combined

    return all_ended

def monitoring_loop():
    """Main loop that continuously processes frames while monitoring is active."""
    global monitoring_active
    while monitoring_active:
        try:
            all_ended = process_frame()
            if all_ended:
                print("All videos ended")
                monitoring_active = False
                break
            time.sleep(0.03)  # Control processing rate to ~30 FPS
        except Exception as e:
            print(f"Error in monitoring loop: {e}")
            monitoring_active = False
            break

def start_monitoring(north_video, east_video, west_video, south_video):
    """Starts the monitoring process."""
    global monitoring_active, monitor_thread, current_frame

    if not all([north_video, east_video, west_video, south_video]):
        return None, "Please upload all 4 video files."

    if monitoring_active:
        return None, "Monitoring is already active. Stop it first."

    if model is None and not initialize_model():
        return None, "Failed to initialize model. Check logs."

    video_paths = {
        "North": north_video,
        "East": east_video,
        "West": west_video,
        "South": south_video
    }

    if not setup_videos(video_paths):
        return None, "Failed to setup videos. Check file paths."

    monitoring_active = True
    current_frame = None
    monitor_thread = threading.Thread(target=monitoring_loop, daemon=True)
    monitor_thread.start()

    return None, "Monitoring started! Live feed is active."

def stop_monitoring():
    """Stops the monitoring process and cleans up resources."""
    global monitoring_active, current_frame
    monitoring_active = False
    if monitor_thread:
        monitor_thread.join(timeout=2)
    for cap in caps.values():
        if cap is not None:
            cap.release()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    final_counts = dict(vehicle_counts) if vehicle_counts else {}
    print("Monitoring stopped. Final counts:", final_counts)

    return None, f"Monitoring stopped. Final counts: {final_counts}"

def get_current_frame():
    """Returns the latest processed frame for the Gradio UI."""
    if current_frame is None:
        blank = np.zeros((720, 1280, 3), dtype=np.uint8)
        cv2.putText(blank, "No video stream active", (400, 360),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
        return Image.fromarray(blank)

    with frame_lock:
        frame_rgb = cv2.cvtColor(current_frame.copy(), cv2.COLOR_BGR2RGB)
        return Image.fromarray(frame_rgb)

def get_current_stats():
    """Returns a formatted string of the current system statistics."""
    if not monitoring_active and not vehicle_counts:
        return "Monitoring is not active. Upload videos and click 'Start Monitoring'."

    stats = "=== VEHICLE COUNTS ===\n"
    stats += f"North: {vehicle_counts.get('North', 0)}\n"
    stats += f"East:  {vehicle_counts.get('East', 0)}\n"
    stats += f"West:  {vehicle_counts.get('West', 0)}\n"
    stats += f"South: {vehicle_counts.get('South', 0)}\n"
    stats += f"TOTAL: {sum(vehicle_counts.values())}\n\n"

    stats += "=== REAL-TIME QUEUES ===\n"
    stats += f"North: {queue_lengths.get('North', 0)}\n"
    stats += f"East:  {queue_lengths.get('East', 0)}\n"
    stats += f"West:  {queue_lengths.get('West', 0)}\n"
    stats += f"South: {queue_lengths.get('South', 0)}\n\n"

    stats += "=== TRAFFIC LIGHT STATUS ===\n"
    if amber_timer > 0:
        stats += f"PHASE: AMBER (Switching)\n"
        stats += f"From: {current_green_direction}\n"
        stats += f"To: {next_green_direction}\n"
        stats += f"Amber Time: {amber_timer / 30:.1f}s / 3.0s\n"
    else:
        stats += f"PHASE: GREEN\n"
        stats += f"Current Green: {current_green_direction}\n"
        stats += f"Green Time: {green_light_timer / 30:.1f}s\n"

    stats += "\nLight States:\n"
    for direction, state in light_states.items():
        if state == "green":
            emoji = "🟢"
        elif state == "amber":
            emoji = "🟡"
        else:
            emoji = "🔴"
        stats += f"{direction}: {emoji} {state.upper()}\n"

    # Throughput metrics
    if start_time is not None:
        elapsed_time = time.time() - start_time
        if elapsed_time > 0:
            stats += "\n=== THROUGHPUT METRICS ===\n"
            total_vehicles = sum(vehicle_counts.values())
            vehicles_per_min = (total_vehicles / elapsed_time) * 60
            stats += f"Total Processed: {total_vehicles}\n"
            stats += f"Processing Rate: {vehicles_per_min:.1f} vehicles/min\n"
            stats += f"Avg Queue: {sum(queue_lengths.values()) / 4:.1f}\n"
            stats += f"Total Cycles: {total_light_cycles}\n"

            if green_duration_history:
                avg_green = sum(green_duration_history) / len(green_duration_history)
                stats += f"Avg Green Duration: {avg_green:.1f}s\n"

    return stats

# --- Gradio User Interface ---
with gr.Blocks(title="Real time Smart city monitoring system using IoT and AI") as demo:
    gr.Markdown("# Real time Smart city monitoring system using IoT and AI ")
    gr.Markdown("Upload four video files, start the system, and observe the real-time vehicle counting and adaptive traffic light control with proper Red-Amber-Green sequencing.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload Videos")
            north_video = gr.File(label="North Direction Video", file_types=["video"])
            east_video = gr.File(label="East Direction Video", file_types=["video"])
            west_video = gr.File(label="West Direction Video", file_types=["video"])
            south_video = gr.File(label="South Direction Video", file_types=["video"])

            gr.Markdown("### 2. Control System")
            with gr.Row():
                start_btn = gr.Button("Start Monitoring", variant="primary")
                stop_btn = gr.Button(" Stop Monitoring", variant="stop")

            status_text = gr.Textbox(label="System Status", interactive=False)

        with gr.Column(scale=3):
            gr.Markdown("### Live Feed")
            live_image = gr.Image(label="Combined Traffic View", height=600, interactive=False)
            stats_text = gr.Textbox(label="Current Statistics", lines=15, interactive=False)

    # Event Handlers
    start_btn.click(
        fn=start_monitoring,
        inputs=[north_video, east_video, west_video, south_video],
        outputs=[live_image, status_text]
    )
    stop_btn.click(
        fn=stop_monitoring,
        outputs=[live_image, status_text]
    )

    # Use gr.Timer for continuous updates
    timer = gr.Timer(value=0.1, active=True)
    timer.tick(
        fn=get_current_frame,
        outputs=[live_image]
    )
    timer.tick(
        fn=get_current_stats,
        outputs=[stats_text]
    )

if __name__ == "__main__":
    demo.launch(debug=True, share=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0e6a6570f1f46180e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using device: cpu
Trying to load yolov8s.pt...
Successfully loaded yolov8s.pt!
Model initialized successfully!
Video setup completed successfully!


ground truth file and detecion file

In [ ]:
!pip install ultralytics opencv-python pandas numpy tqdm matplotlib --quiet

import os, cv2, pandas as pd
from tqdm import tqdm
from matplotlib import pyplot as plt
from matplotlib.widgets import RectangleSelector
from ultralytics import YOLO

# ---------------- CONFIG ----------------
CAMERA_FILES = {
    "North": "/1110127033-preview.mp4",
    "East":  "/3471087857-preview.mp4",
    "West":  "/WhatsApp Video 2025-08-30 at 12.25.04 AM.mp4",
    "South": "/WhatsApp Video 2025-08-30 at 12.25.15 AM.mp4"
}

FRAME_DIR = "frames"
GT_FILE = "ground_truth2.csv"
DET_FILE = "detections.csv"
FRAME_STEP = 15   # Extract every 15th frame
VALID_CLASSES = ["car", "bus", "truck", "motorbike"]

os.makedirs(FRAME_DIR, exist_ok=True)

# ---------------- STEP 1: EXTRACT FRAMES ----------------
for cam, path in CAMERA_FILES.items():
    cam_dir = os.path.join(FRAME_DIR, cam)
    os.makedirs(cam_dir, exist_ok=True)

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_num = 0
    print(f"\nExtracting frames from {cam}...")

    while True:
        ret, frame = cap.read()
        if not ret: break
        if frame_num % FRAME_STEP == 0:
            cv2.imwrite(f"{cam_dir}/frame_{frame_num:06d}.jpg", frame)
        frame_num += 1
    cap.release()
    print(f"Saved {len(os.listdir(cam_dir))} frames for {cam}")

# ---------------- STEP 2: MANUAL ANNOTATION ----------------
columns = ["frame", "camera", "id", "class", "x1", "y1", "x2", "y2"]
gt = pd.DataFrame(columns=columns)

def annotate_image(img, cam, frame_num):
    fig, ax = plt.subplots(figsize=(8,5))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{cam} — Frame {frame_num}\nDraw boxes then close window")
    boxes = []

    def onselect(eclick, erelease):
        x1, y1 = int(eclick.xdata), int(eclick.ydata)
        x2, y2 = int(erelease.xdata), int(erelease.ydata)
        boxes.append([x1, y1, x2, y2])
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        fig.canvas.draw_idle()

    selector = RectangleSelector(
        ax, onselect, useblit=True,
        minspanx=5, minspany=5, spancoords='pixels', interactive=True
    )

    plt.show(block=True)
    return boxes

for cam in CAMERA_FILES.keys():
    cam_dir = os.path.join(FRAME_DIR, cam)
    for f in sorted(os.listdir(cam_dir)):
        frame_num = int(f.split("_")[1].split(".")[0])
        img = cv2.imread(os.path.join(cam_dir, f))
        boxes = annotate_image(img, cam, frame_num)
        for (x1, y1, x2, y2) in boxes:
            vid = input("Enter vehicle ID: ")
            vclass = input(f"Enter class {VALID_CLASSES}: ").lower()
            gt.loc[len(gt)] = [frame_num, cam, vid, vclass, x1, y1, x2, y2]
        gt.to_csv(GT_FILE, index=False)
        print(f"✅ Saved frame {frame_num} to ground truth")

print(f"\n✅ Ground truth saved as {GT_FILE}")

# ---------------- STEP 3: RUN YOLOv8 DETECTION ----------------
model = YOLO("yolov8m.pt")
detection_results = []

for cam in CAMERA_FILES.keys():
    cam_dir = os.path.join(FRAME_DIR, cam)
    for f in sorted(os.listdir(cam_dir)):
        frame_num = int(f.split("_")[1].split(".")[0])
        img = cv2.imread(os.path.join(cam_dir, f))
        results = model(img)
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                if model.names[cls] in VALID_CLASSES:
                    detection_results.append([frame_num, cam, model.names[cls], conf, x1, y1, x2, y2])

det_df = pd.DataFrame(detection_results, columns=["frame","camera","class","confidence","x1","y1","x2","y2"])
det_df.to_csv(DET_FILE, index=False)
print(f"\n✅ Detection file saved as {DET_FILE}")



Extracting frames from North...
Saved 0 frames for North

Extracting frames from East...
Saved 0 frames for East

Extracting frames from West...
Saved 0 frames for West

Extracting frames from South...
Saved 0 frames for South

✅ Ground truth saved as ground_truth2.csv

✅ Detection file saved as detections.csv


In [ ]:
!pip install ultralytics opencv-python pandas numpy tqdm matplotlib --quiet

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ----------------------------------------------------------
# CONFIG
# ----------------------------------------------------------
VIDEO_PATHS = [
    "/content/1110127033-preview.mp4",
    "/content/3471087857-preview.mp4",
    "/content/WhatsApp Video 2025-08-30 at 12.24.49 AM.mp4",
    "/content/WhatsApp Video 2025-08-30 at 12.25.04 AM.mp4"
]

FRAME_ROOT = "frames"
GT_FILE = "ground_truth.csv"
VALID_CLASSES = ["car", "bus", "truck", "motorcycle"]  # COCO classes
FRAME_SKIP = 5  # Extract every Nth frame (adjust based on video FPS)
CONFIDENCE_THRESHOLD = 0.3  # Lower threshold to get more annotations

os.makedirs(FRAME_ROOT, exist_ok=True)

# ----------------------------------------------------------
# OPTION 1: AUTO-LABEL WITH PRETRAINED YOLO (RECOMMENDED)
# ----------------------------------------------------------
def extract_frames_from_video(video_path, camera_id, frame_skip=5):
    """Extract frames from video and save them"""
    logger.info(f"Extracting frames from {Path(video_path).name}...")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logger.error(f"Cannot open video: {video_path}")
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    logger.info(f"Video FPS: {fps}, Total frames: {total_frames}")

    # Create camera directory
    cam_dir = os.path.join(FRAME_ROOT, camera_id)
    os.makedirs(cam_dir, exist_ok=True)

    frame_count = 0
    saved_count = 0

    pbar = tqdm(total=total_frames, desc=f"Extracting frames ({camera_id})")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Save every Nth frame
        if frame_count % frame_skip == 0:
            frame_path = os.path.join(cam_dir, f"frame_{frame_count:06d}.jpg")
            cv2.imwrite(frame_path, frame)
            saved_count += 1

        frame_count += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    logger.info(f"Extracted {saved_count} frames from {total_frames} total frames")
    return saved_count


def auto_label_with_yolo(confidence_threshold=0.3):
    """Use pretrained YOLO to generate ground truth annotations"""
    logger.info("Loading pretrained YOLOv8 model...")
    model = YOLO("yolov8m.pt")  # Pretrained COCO model

    # COCO class mapping (relevant vehicle classes)
    coco_vehicle_map = {
        2: "car",
        5: "bus",
        7: "truck",
        3: "motorcycle"  # Maps to motorbike
    }

    annotations = []

    # Process all camera directories
    camera_dirs = [d for d in os.listdir(FRAME_ROOT) if os.path.isdir(os.path.join(FRAME_ROOT, d))]

    logger.info(f"Found {len(camera_dirs)} camera directories")

    for camera_id in camera_dirs:
        cam_path = os.path.join(FRAME_ROOT, camera_id)
        frame_files = sorted([f for f in os.listdir(cam_path) if f.endswith('.jpg')])

        logger.info(f"Processing {len(frame_files)} frames from camera '{camera_id}'...")

        for frame_file in tqdm(frame_files, desc=f"Labeling {camera_id}"):
            frame_path = os.path.join(cam_path, frame_file)
            frame_num = int(frame_file.split('_')[1].split('.')[0])

            # Run detection
            results = model.predict(
                source=frame_path,
                conf=confidence_threshold,
                classes=list(coco_vehicle_map.keys()),  # Only detect vehicles
                verbose=False
            )

            # Extract detections
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes

                for box in boxes:
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                    # Map to our class names
                    class_name = coco_vehicle_map.get(cls_id, "unknown")
                    if class_name == "motorcycle":
                        class_name = "motorbike"  # Standardize naming

                    annotations.append({
                        'camera': camera_id,
                        'frame': frame_num,
                        'class': class_name,
                        'x1': int(x1),
                        'y1': int(y1),
                        'x2': int(x2),
                        'y2': int(y2),
                        'confidence': round(conf, 3)
                    })

    return annotations


def save_ground_truth(annotations, output_file=GT_FILE):
    """Save annotations to CSV"""
    if not annotations:
        logger.error("No annotations found! Check your videos and frame extraction.")
        return

    df = pd.DataFrame(annotations)
    df = df.sort_values(['camera', 'frame'])
    df.to_csv(output_file, index=False)

    logger.info(f"\n{'='*50}")
    logger.info(f"Ground truth saved to: {output_file}")
    logger.info(f"Total annotations: {len(df)}")
    logger.info(f"\nClass distribution:")
    print(df['class'].value_counts())
    logger.info(f"\nFrames per camera:")
    print(df.groupby('camera')['frame'].nunique())
    logger.info(f"{'='*50}\n")


# ----------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------
def create_ground_truth_from_videos():
    """Complete pipeline: extract frames + auto-label"""

    # Step 1: Extract frames from all videos
    logger.info("="*50)
    logger.info("STEP 1: EXTRACTING FRAMES")
    logger.info("="*50)

    for i, video_path in enumerate(VIDEO_PATHS):
        if not os.path.exists(video_path):
            logger.warning(f"Video not found: {video_path}")
            continue

        camera_id = f"cam_{i+1}"  # Or use video filename
        extract_frames_from_video(video_path, camera_id, frame_skip=FRAME_SKIP)

    # Step 2: Auto-label frames with pretrained YOLO
    logger.info("\n" + "="*50)
    logger.info("STEP 2: AUTO-LABELING WITH YOLO")
    logger.info("="*50)

    annotations = auto_label_with_yolo(confidence_threshold=CONFIDENCE_THRESHOLD)

    # Step 3: Save ground truth
    logger.info("\n" + "="*50)
    logger.info("STEP 3: SAVING GROUND TRUTH")
    logger.info("="*50)

    save_ground_truth(annotations)

    logger.info("✅ Ground truth generation complete!")


# ----------------------------------------------------------
# OPTION 2: MANUAL LABELING HELPER (IF YOU WANT TO LABEL MANUALLY)
# ----------------------------------------------------------
def create_manual_labeling_template():
    """Create a template CSV for manual labeling"""
    logger.info("Creating manual labeling template...")

    frames_data = []

    camera_dirs = [d for d in os.listdir(FRAME_ROOT) if os.path.isdir(os.path.join(FRAME_ROOT, d))]

    for camera_id in camera_dirs:
        cam_path = os.path.join(FRAME_ROOT, camera_id)
        frame_files = sorted([f for f in os.listdir(cam_path) if f.endswith('.jpg')])

        for frame_file in frame_files:
            frame_num = int(frame_file.split('_')[1].split('.')[0])
            frames_data.append({
                'camera': camera_id,
                'frame': frame_num,
                'class': '',  # To be filled manually
                'x1': '',
                'y1': '',
                'x2': '',
                'y2': '',
                'confidence': 1.0
            })

    df = pd.DataFrame(frames_data)
    template_file = "manual_labeling_template.csv"
    df.to_csv(template_file, index=False)

    logger.info(f"Template saved to: {template_file}")
    logger.info("Fill in the class and bbox coordinates, then rename to 'ground_truth.csv'")


# ----------------------------------------------------------
# OPTION 3: LOAD EXISTING LABELS (If you have labels in another format)
# ----------------------------------------------------------
def convert_yolo_labels_to_groundtruth(labels_dir):
    """Convert YOLO format labels to ground truth CSV"""
    logger.info(f"Converting YOLO labels from: {labels_dir}")

    CLASS_NAMES = ["car", "bus", "truck", "motorbike"]
    annotations = []

    for label_file in Path(labels_dir).rglob("*.txt"):
        # Parse filename to get camera and frame
        # Assuming format: cam_X_NNNNNN.txt
        parts = label_file.stem.split('_')
        camera_id = f"{parts[0]}_{parts[1]}"
        frame_num = int(parts[2])

        # Get corresponding image to get dimensions
        img_file = label_file.parent.parent / "images" / label_file.parent.name / f"{label_file.stem}.jpg"
        if not img_file.exists():
            continue

        img = cv2.imread(str(img_file))
        h, w = img.shape[:2]

        # Read YOLO format labels
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                cls_id, x_center, y_center, width, height = map(float, parts)

                # Convert to absolute coordinates
                x1 = int((x_center - width/2) * w)
                y1 = int((y_center - height/2) * h)
                x2 = int((x_center + width/2) * w)
                y2 = int((y_center + height/2) * h)

                annotations.append({
                    'camera': camera_id,
                    'frame': frame_num,
                    'class': CLASS_NAMES[int(cls_id)],
                    'x1': x1,
                    'y1': y1,
                    'x2': x2,
                    'y2': y2,
                    'confidence': 1.0
                })

    return annotations


# ----------------------------------------------------------
# RUN THE PIPELINE
# ----------------------------------------------------------
if __name__ == "__main__":
    print("\n" + "="*60)
    print("  GROUND TRUTH GENERATION FOR TRAFFIC DETECTION")
    print("="*60 + "\n")

    print("Choose an option:")
    print("1. Auto-generate ground truth from videos (RECOMMENDED)")
    print("2. Create manual labeling template")
    print("3. Check existing ground truth file")
    print()

    # For automatic execution in Colab, use option 1
    choice = "1"  # Change this if needed

    if choice == "1":
        create_ground_truth_from_videos()

    elif choice == "2":
        # First extract frames if not done
        for i, video_path in enumerate(VIDEO_PATHS):
            if os.path.exists(video_path):
                camera_id = f"cam_{i+1}"
                extract_frames_from_video(video_path, camera_id, frame_skip=FRAME_SKIP)
        create_manual_labeling_template()

    elif choice == "3":
        if os.path.exists(GT_FILE):
            df = pd.read_csv(GT_FILE)
            logger.info(f"Existing ground truth file found: {GT_FILE}")
            logger.info(f"Total annotations: {len(df)}")
            logger.info(f"\nClass distribution:")
            print(df['class'].value_counts())
            logger.info(f"\nSample rows:")
            print(df.head(10))
        else:
            logger.error(f"Ground truth file not found: {GT_FILE}")
            logger.info("Please run option 1 or 2 first!")

    else:
        logger.error("Invalid choice!")

    print("\n" + "="*60)
    print("  DONE!")
    print("="*60)


  GROUND TRUTH GENERATION FOR TRAFFIC DETECTION

Choose an option:
1. Auto-generate ground truth from videos (RECOMMENDED)
2. Create manual labeling template
3. Check existing ground truth file



Extracting frames (cam_4): 100%|██████████| 515/515 [00:00<00:00, 1240.16it/s]
Labeling West: 0it [00:00, ?it/s]
Labeling cam_3: 100%|██████████| 57/57 [00:01<00:00, 34.43it/s]
Labeling South: 0it [00:00, ?it/s]
Labeling cam_2: 100%|██████████| 105/105 [00:03<00:00, 30.00it/s]
Labeling East: 0it [00:00, ?it/s]
Labeling North: 0it [00:00, ?it/s]
Labeling cam_1: 100%|██████████| 135/135 [00:03<00:00, 37.54it/s]

class
car          4483
truck         636
motorbike     612
bus           343
Name: count, dtype: int64
camera
cam_1    135
cam_2    105
cam_3     57
cam_4    103
Name: frame, dtype: int64

  DONE!


In [ ]:
"""
Multi-Camera Vehicle Detection and Evaluation System (FIXED)
Works with auto-generated ground truth from the first script.
"""

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm import tqdm
from typing import Dict, List, Tuple

# ==================== CONFIGURATION ====================
CAMERA_FILES = {
    "cam_1": "/content/1110127033-preview.mp4",
    "cam_2": "/content/3471087857-preview.mp4",
    "cam_3": "/content/WhatsApp Video 2025-08-30 at 12.24.49 AM.mp4",
    "cam_4": "/content/WhatsApp Video 2025-08-30 at 12.25.04 AM.mp4"
}

FRAME_ROOT = "frames"
GT_FILE = "ground_truth.csv"
DET_FILE = "detections.csv"
FRAME_SKIP = 5  # Must match the first script
VALID_CLASSES = ["car", "bus", "truck", "motorcycle", "motorbike"]
IOU_THRESHOLD = 0.5
CONFIDENCE_THRESHOLD = 0.25

# COCO class mapping for YOLOv8
COCO_VEHICLE_MAP = {
    2: "car",
    5: "bus",
    7: "truck",
    3: "motorcycle"
}

os.makedirs(FRAME_ROOT, exist_ok=True)


# ==================== YOLO DETECTION ON FRAMES ====================
def run_yolo_on_frames(model_name: str = "yolov8m.pt") -> pd.DataFrame:
    """
    Run YOLOv8 detection on extracted frames (not videos).
    This ensures frame numbers match exactly with ground truth.
    """
    print(f"\nLoading YOLO model: {model_name}")
    model = YOLO(model_name)

    detection_results = []

    # Process each camera directory
    camera_dirs = [d for d in os.listdir(FRAME_ROOT)
                   if os.path.isdir(os.path.join(FRAME_ROOT, d))]

    print(f"\nFound {len(camera_dirs)} camera directories")

    for camera_id in camera_dirs:
        cam_path = os.path.join(FRAME_ROOT, camera_id)
        frame_files = sorted([f for f in os.listdir(cam_path) if f.endswith('.jpg')])

        print(f"\nProcessing {len(frame_files)} frames from {camera_id}...")

        for frame_file in tqdm(frame_files, desc=f"Detecting {camera_id}"):
            frame_path = os.path.join(cam_path, frame_file)
            frame_num = int(frame_file.split('_')[1].split('.')[0])

            # Read image
            img = cv2.imread(frame_path)
            if img is None:
                continue

            # Run detection
            results = model.predict(
                source=img,
                conf=CONFIDENCE_THRESHOLD,
                classes=list(COCO_VEHICLE_MAP.keys()),  # Only detect vehicles
                verbose=False
            )

            # Extract detections
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes

                for box in boxes:
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                    # Map to class names
                    class_name = COCO_VEHICLE_MAP.get(cls_id, "unknown")
                    if class_name == "motorcycle":
                        class_name = "motorbike"  # Standardize naming

                    detection_results.append({
                        'camera': camera_id,
                        'frame': frame_num,
                        'class': class_name,
                        'x1': int(x1),
                        'y1': int(y1),
                        'x2': int(x2),
                        'y2': int(y2),
                        'confidence': round(conf, 3)
                    })

    det_df = pd.DataFrame(detection_results)

    if len(det_df) > 0:
        det_df = det_df.sort_values(['camera', 'frame'])
        det_df.to_csv(DET_FILE, index=False)
        print(f"\n✓ Saved {len(det_df)} detections to {DET_FILE}")
        print(f"\nDetection class distribution:")
        print(det_df['class'].value_counts())
    else:
        print("\n⚠️  No detections found!")

    return det_df


# ==================== EVALUATION METRICS ====================
def calculate_iou(boxA: List, boxB: List) -> float:
    """Calculate Intersection over Union between two boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_area = max(0, xB - xA) * max(0, yB - yA)

    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union_area = boxA_area + boxB_area - inter_area

    return inter_area / (union_area + 1e-6)


def evaluate_detections(gt: pd.DataFrame, det_df: pd.DataFrame) -> Dict:
    """
    Evaluate detection performance against ground truth.
    Fixed to handle auto-generated ground truth format.
    """
    print("\nEvaluating detections...")
    print(f"Ground truth annotations: {len(gt)}")
    print(f"Detections: {len(det_df)}")

    if len(gt) == 0:
        print("⚠️  No ground truth annotations!")
        return {}

    if len(det_df) == 0:
        print("⚠️  No detections!")
        return {}

    ious = []
    tp = 0  # True Positives
    fp = 0  # False Positives
    fn = 0  # False Negatives

    matched_detections = set()

    # For each ground truth box, find best matching detection
    for idx, gt_row in tqdm(gt.iterrows(), total=len(gt), desc="Matching GT to Detections"):
        # Filter detections for same camera and frame
        dets = det_df[
            (det_df['camera'] == gt_row['camera']) &
            (det_df['frame'] == gt_row['frame'])
        ]

        if len(dets) == 0:
            fn += 1
            continue

        gt_box = [gt_row['x1'], gt_row['y1'], gt_row['x2'], gt_row['y2']]
        gt_class = gt_row['class'].lower().strip()

        # Normalize class names
        if gt_class == "motorcycle":
            gt_class = "motorbike"

        best_iou = 0
        best_det_idx = None

        # Find best matching detection
        for det_idx, det_row in dets.iterrows():
            det_box = [det_row['x1'], det_row['y1'], det_row['x2'], det_row['y2']]
            det_class = det_row['class'].lower().strip()

            # Normalize class names
            if det_class == "motorcycle":
                det_class = "motorbike"

            # Check if classes match
            if gt_class != det_class:
                continue

            # Calculate IoU
            iou_score = calculate_iou(gt_box, det_box)

            if iou_score > best_iou:
                best_iou = iou_score
                best_det_idx = det_idx

        # Check if match is good enough
        if best_iou >= IOU_THRESHOLD and best_det_idx is not None:
            tp += 1
            ious.append(best_iou)
            matched_detections.add(best_det_idx)
        else:
            fn += 1

    # False positives: detections that don't match any GT
    fp = len(det_df) - len(matched_detections)

    # Calculate metrics
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    mean_iou = np.mean(ious) if ious else 0.0

    metrics = {
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1-Score": round(f1, 3),
        "mIoU": round(mean_iou, 3),
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "Total GT": len(gt),
        "Total Detections": len(det_df),
        "Matched Detections": len(matched_detections)
    }

    # Per-class metrics
    print("\n" + "="*60)
    print("Per-Class Performance:")
    print("="*60)

    for cls in VALID_CLASSES:
        gt_cls = gt[gt['class'].str.lower() == cls.lower()]
        det_cls = det_df[det_df['class'].str.lower() == cls.lower()]

        if len(gt_cls) > 0:
            cls_tp = 0
            for _, gt_row in gt_cls.iterrows():
                dets = det_cls[
                    (det_cls['camera'] == gt_row['camera']) &
                    (det_cls['frame'] == gt_row['frame'])
                ]

                gt_box = [gt_row['x1'], gt_row['y1'], gt_row['x2'], gt_row['y2']]

                for _, det_row in dets.iterrows():
                    det_box = [det_row['x1'], det_row['y1'], det_row['x2'], det_row['y2']]
                    if calculate_iou(gt_box, det_box) >= IOU_THRESHOLD:
                        cls_tp += 1
                        break

            cls_recall = cls_tp / len(gt_cls)
            print(f"{cls:12s}: GT={len(gt_cls):4d}, Det={len(det_cls):4d}, TP={cls_tp:4d}, Recall={cls_recall:.3f}")

    return metrics


def visualize_metrics(metrics: Dict) -> None:
    """Create visualization of evaluation metrics."""
    if not metrics:
        print("No metrics to visualize!")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Performance metrics
    ax1 = axes[0, 0]
    perf_metrics = ['Precision', 'Recall', 'F1-Score', 'mIoU']
    perf_values = [metrics.get(m, 0) for m in perf_metrics]

    colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
    bars1 = ax1.bar(perf_metrics, perf_values, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_ylim(0, 1.0)
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Detection Performance Metrics', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

    # 2. Confusion matrix
    ax2 = axes[0, 1]
    confusion_data = {
        'TP': metrics.get('True Positives', 0),
        'FP': metrics.get('False Positives', 0),
        'FN': metrics.get('False Negatives', 0)
    }

    colors2 = ['#2ecc71', '#e74c3c', '#f39c12']
    bars2 = ax2.bar(confusion_data.keys(), confusion_data.values(),
                     color=colors2, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax2.set_title('Detection Results', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')

    # 3. Summary statistics
    ax3 = axes[1, 0]
    ax3.axis('off')

    summary_text = f"""
    EVALUATION SUMMARY
    {'='*40}

    Total Ground Truth:     {metrics.get('Total GT', 0):,}
    Total Detections:       {metrics.get('Total Detections', 0):,}
    Matched Detections:     {metrics.get('Matched Detections', 0):,}

    True Positives:         {metrics.get('True Positives', 0):,}
    False Positives:        {metrics.get('False Positives', 0):,}
    False Negatives:        {metrics.get('False Negatives', 0):,}

    Precision:              {metrics.get('Precision', 0):.3f}
    Recall:                 {metrics.get('Recall', 0):.3f}
    F1-Score:               {metrics.get('F1-Score', 0):.3f}
    Mean IoU:               {metrics.get('mIoU', 0):.3f}
    """

    ax3.text(0.1, 0.5, summary_text, transform=ax3.transAxes,
            fontsize=11, verticalalignment='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

    # 4. Precision-Recall visualization
    ax4 = axes[1, 1]
    precision = metrics.get('Precision', 0)
    recall = metrics.get('Recall', 0)

    ax4.scatter([recall], [precision], s=500, c='red', alpha=0.6, edgecolors='black', linewidths=2)
    ax4.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.set_xlabel('Recall', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Precision', fontsize=12, fontweight='bold')
    ax4.set_title('Precision-Recall Point', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend()

    ax4.text(recall, precision + 0.05, f'({recall:.3f}, {precision:.3f})',
            ha='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n✓ Metrics visualization saved as 'evaluation_metrics.png'")


# ==================== MAIN EXECUTION ====================
def main():
    """Main execution pipeline."""
    print("="*60)
    print("Multi-Camera Vehicle Detection Evaluation")
    print("="*60)

    # Check if ground truth exists
    if not os.path.exists(GT_FILE):
        print(f"\n⚠️  Ground truth file not found: {GT_FILE}")
        print("Please run the first script to generate ground truth first!")
        return

    # Load ground truth
    print(f"\nLoading ground truth from {GT_FILE}...")
    gt = pd.read_csv(GT_FILE)
    print(f"✓ Loaded {len(gt)} ground truth annotations")
    print(f"\nGround truth class distribution:")
    print(gt['class'].value_counts())
    print(f"\nFrames per camera:")
    print(gt.groupby('camera')['frame'].nunique())

    # Check if frames exist
    if not os.path.exists(FRAME_ROOT) or len(os.listdir(FRAME_ROOT)) == 0:
        print(f"\n⚠️  Frames directory not found or empty: {FRAME_ROOT}")
        print("Please run the first script to extract frames first!")
        return

    # Run YOLO detection
    print("\n" + "="*60)
    print("Running YOLO detection on frames...")
    print("="*60)
    det_df = run_yolo_on_frames()

    if len(det_df) == 0:
        print("\n⚠️  No detections generated. Cannot evaluate.")
        return

    # Evaluate results
    print("\n" + "="*60)
    print("Evaluating detection performance...")
    print("="*60)
    metrics = evaluate_detections(gt, det_df)

    if not metrics:
        print("\n⚠️  Evaluation failed!")
        return

    # Display results
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)
    for key, value in metrics.items():
        print(f"{key:25s}: {value}")

    # Save results
    results_df = pd.DataFrame([metrics]).T.rename(columns={0: "Value"})
    results_df.to_csv("evaluation_results.csv")
    print("\n✓ Results saved to 'evaluation_results.csv'")

    # Visualize
    visualize_metrics(metrics)

    print("\n" + "="*60)
    print("Evaluation completed successfully!")
    print("="*60)


if __name__ == "__main__":
    main()

Multi-Camera Vehicle Detection Evaluation

Loading ground truth from ground_truth.csv...
✓ Loaded 6074 ground truth annotations

Ground truth class distribution:
class
car          4483
truck         636
motorbike     612
bus           343
Name: count, dtype: int64

Frames per camera:
camera
cam_1    135
cam_2    105
cam_3     57
cam_4    103
Name: frame, dtype: int64

Running YOLO detection on frames...

Loading YOLO model: yolov8m.pt

Found 8 camera directories

Processing 0 frames from West...


Detecting West: 0it [00:00, ?it/s]



Processing 57 frames from cam_3...


Detecting cam_3: 100%|██████████| 57/57 [00:01<00:00, 38.52it/s]



Processing 0 frames from South...


...


Processing 135 frames from cam_1...


Detecting cam_1: 100%|██████████| 135/135 [00:03<00:00, 34.55it/s]



✓ Saved 6647 detections to detections.csv

Detection class distribution:
class
car          4886
truck         710
motorbike     663
bus           388
Name: count, dtype: int64

Evaluating detection performance...

Evaluating detections...
Ground truth annotations: 6074
Detections: 6647


Matching GT to Detections: 100%|██████████| 6074/6074 [00:15<00:00, 382.91it/s]



Per-Class Performance:
car         : GT=4483, Det=4886, TP=4483, Recall=1.000
bus         : GT= 343, Det= 388, TP= 343, Recall=1.000
truck       : GT= 636, Det= 710, TP= 636, Recall=1.000
motorbike   : GT= 612, Det= 663, TP= 612, Recall=1.000

EVALUATION RESULTS
Precision                : 0.914
Recall                   : 1.0
F1-Score                 : 0.955
mIoU                     : 1.0
True Positives           : 6074
False Positives          : 573
False Negatives          : 0
Total GT                 : 6074
Total Detections         : 6647
Matched Detections       : 6074

✓ Results saved to 'evaluation_results.csv'

✓ Metrics visualization saved as 'evaluation_metrics.png'

Evaluation completed successfully!
